## 1) Kaggle set-up

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/exoplanet-detection-challenge/sample_submission.csv
/kaggle/input/competitions/exoplanet-detection-challenge/train.csv
/kaggle/input/competitions/exoplanet-detection-challenge/test.csv


## 2) Imports

In [2]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

## 3) Load and Check Data

In [3]:
train = pd.read_csv('/kaggle/input/competitions/exoplanet-detection-challenge/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/exoplanet-detection-challenge/test.csv')
sub   = pd.read_csv('/kaggle/input/competitions/exoplanet-detection-challenge/sample_submission.csv')

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Label distribution:\n", train['label'].value_counts())

Train shape: (9000, 27)
Test shape: (3000, 26)
Label distribution:
 label
0    6877
1    2123
Name: count, dtype: int64


## 4) Feature Engineering 

In [4]:

leaky_planet_cols = ['planet_radius_re', 'rp_rs_ratio', 'semi_major_au','eccentricity',
                           'inclination_deg', 'orbital_velocity_kms','planet_eq_temp_k',
                           'orbital_period_d']

le=LabelEncoder()
le.fit(train['spectral_type'].fillna('Unknown'))

fill_vals={'stellar_metallicity':train['stellar_metallicity'].median(),
           'stellar_rot_period_d': train['stellar_rot_period_d'].median(),
           'transit_duration_hr':  train['transit_duration_hr'].median(),}

def engineer(df,le,fill_vals):
    df = df.copy()
    
    # Drop planet-derived columns (only exist after planet confirmed — leaky)
    df = df.drop(columns=[c for c in leaky_planet_cols if c in df.columns])
    
    # Encode spectral type: F G K M → 0 1 2 3
    df['spectral_enc'] = le.transform(df['spectral_type'].fillna('Unknown'))
    df = df.drop(columns=['star_id', 'spectral_type'])
    
    # Fill missing stellar values
    df['stellar_metallicity']  = df['stellar_metallicity'].fillna(fill_vals['stellar_metallicity'])
    df['stellar_rot_period_d'] = df['stellar_rot_period_d'].fillna(fill_vals['stellar_rot_period_d'])
    df['transit_duration_hr']  = df['transit_duration_hr'].fillna(fill_vals['transit_duration_hr'])  # no transit = 0
    
    # Interaction features
    df['snr_x_transits']   = df['transit_snr'] * df['n_transits_observed']
    df['depth_x_transits'] = df['transit_depth_ppm'] * df['n_transits_observed']
    df['depth_x_dur']      = df['transit_depth_ppm'] * df['transit_duration_hr']
    df['log_depth']        = np.log1p(df['transit_depth_ppm'])
    df['log_transits']     = np.log1p(df['n_transits_observed'])
    df['log_flux']         = np.log1p(df['flux_variability_index'])
    df['snr_per_transit']  = df['transit_snr'] / (df['n_transits_observed'] + 1)
    df['lum_per_mass']     = df['stellar_luminosity'] / (df['stellar_mass_sm'] + 1e-6)
    df['radius_per_mass']  = df['stellar_radius_sr'] / (df['stellar_mass_sm'] + 1e-6)
    df['teff_logg']        = df['stellar_teff_k'] / (df['stellar_log_g'] + 1e-6)
    
    return df

X      = engineer(train.drop(columns=['label']),le,fill_vals)
y      = train['label']
X_test = engineer(test,le,fill_vals)

print("Feature matrix:", X.shape)
print("Features:", list(X.columns))



Feature matrix: (9000, 27)
Features: ['stellar_radius_sr', 'stellar_mass_sm', 'stellar_teff_k', 'stellar_log_g', 'stellar_luminosity', 'stellar_metallicity', 'stellar_rot_period_d', 'stellar_noise_ppm', 'impact_parameter', 'transit_depth_ppm', 'transit_duration_hr', 'n_transits_observed', 'transit_snr', 'flux_variability_index', 'log_period', 'log_snr', 'spectral_enc', 'snr_x_transits', 'depth_x_transits', 'depth_x_dur', 'log_depth', 'log_transits', 'log_flux', 'snr_per_transit', 'lum_per_mass', 'radius_per_mass', 'teff_logg']


## 5) Define Models

In [5]:

scale = (y == 0).sum() / (y == 1).sum()  # 6877/2123 ≈ 3.24
print(f"Class imbalance ratio: {scale:.2f}")

xgb_model = xgb.XGBClassifier(
    n_estimators=1000,
    learning_rate=0.02,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=scale,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

lgb_model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.02,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=20,
    reg_alpha=0.1,
    reg_lambda=1.0,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=12,
    min_samples_leaf=3,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

models = [xgb_model, lgb_model, rf_model]
model_names = ['XGBoost', 'LightGBM', 'RandomForest']

Class imbalance ratio: 3.24


## 6) OOF Training Loop

In [6]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

oof_preds  = np.zeros((len(X), len(models)))
test_preds = np.zeros((len(X_test), len(models)))

for mi, (model, name) in enumerate(zip(models, model_names)):
    print(f"\n--- {name} ---")
    tp = np.zeros(len(X_test))
    
    for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        
        model.fit(X_tr, y_tr)
        
        val_prob = model.predict_proba(X_val)[:, 1]
        oof_preds[val_idx, mi] = val_prob
        
        fold_acc = ((val_prob > 0.5).astype(int) == y_val).mean()
        print(f"  Fold {fold+1}: accuracy = {fold_acc:.4f}")
        
        tp += model.predict_proba(X_test)[:, 1]
    
    test_preds[:, mi] = tp / 10
    oof_acc = ((oof_preds[:, mi] > 0.5).astype(int) == y).mean()
    print(f"  {name} OOF accuracy: {oof_acc:.4f}")


--- XGBoost ---
  Fold 1: accuracy = 1.0000
  Fold 2: accuracy = 1.0000
  Fold 3: accuracy = 0.9989
  Fold 4: accuracy = 1.0000
  Fold 5: accuracy = 1.0000
  Fold 6: accuracy = 1.0000
  Fold 7: accuracy = 1.0000
  Fold 8: accuracy = 1.0000
  Fold 9: accuracy = 1.0000
  Fold 10: accuracy = 1.0000
  XGBoost OOF accuracy: 0.9999

--- LightGBM ---
  Fold 1: accuracy = 1.0000
  Fold 2: accuracy = 1.0000
  Fold 3: accuracy = 1.0000
  Fold 4: accuracy = 1.0000
  Fold 5: accuracy = 1.0000
  Fold 6: accuracy = 1.0000
  Fold 7: accuracy = 1.0000
  Fold 8: accuracy = 1.0000
  Fold 9: accuracy = 1.0000
  Fold 10: accuracy = 1.0000
  LightGBM OOF accuracy: 1.0000

--- RandomForest ---
  Fold 1: accuracy = 1.0000
  Fold 2: accuracy = 1.0000
  Fold 3: accuracy = 1.0000
  Fold 4: accuracy = 1.0000
  Fold 5: accuracy = 1.0000
  Fold 6: accuracy = 1.0000
  Fold 7: accuracy = 1.0000
  Fold 8: accuracy = 1.0000
  Fold 9: accuracy = 1.0000
  Fold 10: accuracy = 1.0000
  RandomForest OOF accuracy: 1.0000


## 7) Ensemble and Submission

In [7]:
final_prob = test_preds.mean(axis=1)
final_pred = (final_prob > 0.5).astype(int)

# Overall OOF accuracy
oof_ensemble = (oof_preds.mean(axis=1) > 0.5).astype(int)
print(f"\nEnsemble OOF accuracy: {(oof_ensemble == y).mean():.4f}")
print(f"Predicted label distribution: {pd.Series(final_pred).value_counts().to_dict()}")

# Build submission
sub['label'] = final_pred
sub.to_csv('/kaggle/working/submission.csv', index=False)
print("\nsubmission.csv saved to /kaggle/working/")
sub.head(10)


Ensemble OOF accuracy: 1.0000
Predicted label distribution: {0: 2293, 1: 707}

submission.csv saved to /kaggle/working/


,star_id,label
0,STR-011368,0
1,STR-005379,0
2,STR-004743,1
3,STR-007357,0
4,STR-010554,0
5,STR-002573,1
6,STR-004111,0
7,STR-002827,0
8,STR-009245,0
9,STR-007693,0
